# Lab 10: Memory Patterns for Conversational LLMs

**Week 2 · LangChain memory · engineer-to-engineer**

**Theme.** Build and compare three ways to give a chat assistant *memory* (a **full buffer**, a **rolling summary**, and an **entity or fact store**), then reason about the trade-offs: recall against token cost and latency, privacy, and when *not* to use memory at all.

**Mental model (anchor it to what you already trust).** An LLM call is stateless, like an HTTP request with no session. "Memory" is nothing magic: it is a list of messages *you* own, curate, and re-send on every turn. Buffer keeps the whole list. Summary compresses the old part. Entity memory keeps a side table of extracted facts. That is the entire idea; the rest is engineering.

**Local by default, live when you want it.** Every cell runs offline with a deterministic scripted model, so checks are reproducible with no server and no keys. One flag, `MODE`, selects the backend: `"offline"` (the default), `"lmstudio"`, or `"ollama"`. Both live options are local OpenAI-compatible servers, so a single code path drives both. A native client such as `langchain-ollama` is a valid alternative to the OpenAI-compatible endpoint, but it is not needed here.


## Objectives

By the end of this lab you will be able to:

1. Implement **buffer**, **summary**, and **entity** memory using stable `langchain_core` message primitives (no deprecated memory classes).
2. Measure how **assembled-context size** and **approximate token cost** differ across the three patterns for the same conversation.
3. Verify **fact recall** that depends on prior turns, and show where windowing and summarization *lose* facts.
4. Apply **PII redaction** before storage and explain, with a working baseline, when *not* to use memory.


> ### CURRENCY FLAG (read before you teach)
>
> LangChain's classic memory classes (`ConversationBufferMemory`, `ConversationSummaryMemory`, `ConversationEntityMemory`, `ConversationChain`) were **deprecated in 0.3.1 and removed from the main `langchain` package** in the 1.x line. On the cohort stack (`langchain==1.3.13`) `from langchain.memory import ConversationBufferMemory` raises `ModuleNotFoundError`.
>
> Their first replacement, `RunnableWithMessageHistory`, is **also deprecated** in 1.x (it now points you to LangGraph persistence). Rather than teach an API that is on its third deprecation in two years, this lab builds memory directly on **stable core primitives** (`HumanMessage` / `AIMessage` / `SystemMessage` / `trim_messages`). That teaches the durable concept and produces **zero deprecation warnings**.
>
> Landscape to mention verbally: for production multi-user memory, LangChain now points to **LangGraph checkpointers**. That is out of scope for a "LangChain basics" day, but name it so learners know where the road goes.


---
## Part A: Setup, data, and helpers *(instructor I-do; nothing to implement here)*

Run these four cells top to bottom. They import the stack, load the synthetic support chat, define the model backend selector (offline scripted, LM Studio, or Ollama), the metrics, and a soft `check()` that never crashes the notebook.

In [ ]:
%pip install -r requirements.txt

In [ ]:
# A1: imports and versions
from __future__ import annotations
import os, re, json, time, importlib.metadata as md

from langchain_core.messages import (
    BaseMessage, HumanMessage, AIMessage, SystemMessage, trim_messages,
)
from langchain_core.language_models.fake_chat_models import FakeListChatModel
from rapidfuzz import fuzz

for pkg in ["langchain", "langchain-core", "langchain-openai", "rapidfuzz"]:
    try:
        print(f"{pkg:16s} {md.version(pkg)}")
    except md.PackageNotFoundError:
        print(f"{pkg:16s} (not installed)")

In [ ]:
# A2: the synthetic conversation (self-contained; regenerated here, no cross-lab deps)
SYSTEM_PROMPT = "You are a concise customer-success assistant. If unsure, ask a clarifying question."

CHAT = {
    "turns": [
        {"role": "user",      "text": "Hi, I'm Jordan. I use your video editor on a MacBook Pro."},
        {"role": "assistant", "text": "Hi Jordan! How can I help with the video editor today?"},
        {"role": "user",      "text": "My export keeps failing for 4K timelines. I'm on app v2.3 with an M1 Pro."},
        {"role": "assistant", "text": "Got it. Do you see any error code or crash report?"},
        {"role": "user",      "text": "No code, but the app quits near the end. Also, I prefer H.265 exports with 20 Mbps."},
        {"role": "assistant", "text": "Thanks. I'll keep that preference in mind."},
        {"role": "user",      "text": "My last support ticket was #A-8821 if that helps."},
        {"role": "assistant", "text": "Thanks, I'll reference ticket A-8821."},
        {"role": "user",      "text": "What export settings did I prefer again? And what was my last ticket ID?"},
    ],
    "eval": {
        "user": "Jordan", "device": "MacBook Pro",
        "prefs_tokens": ["H.265", "20 Mbps"], "ticket": "A-8821",
        "expected_answer_contains": ["H.265", "20 Mbps", "A-8821"],
    },
}
TURNS   = CHAT["turns"]
EV      = CHAT["eval"]
HISTORY = TURNS[:-1]          # everything before the final question
FINAL_Q = TURNS[-1]["text"]   # "What export settings did I prefer again? ..."
print(f"{len(TURNS)} turns loaded; final question:\n  {FINAL_Q}")

In [ ]:
# A3: model backend selector and metrics
# MODE picks where every model() call goes. Default "offline" needs no server and is
# fully reproducible, which is why the checks are stable. The two live backends are
# local OpenAI-compatible servers and share one client (ChatOpenAI), differing only in
# base_url, default model, and api_key placeholder.
#   "offline"   deterministic scripted replies (no server, no keys)
#   "lmstudio"  local LM Studio server (default http://localhost:1234/v1)
#   "ollama"    local Ollama server    (default http://localhost:11434/v1)
MODE = "ollama"

# Per-backend connection settings. Env vars override, so you rarely edit this cell.
BACKENDS = {
    "lmstudio": {
        "base_url": os.getenv("LMSTUDIO_BASE_URL", "http://localhost:1234/v1"),
        "model":    os.getenv("LMSTUDIO_MODEL", "local-model"),
        "api_key":  os.getenv("LMSTUDIO_API_KEY", "lm-studio"),
    },
    "ollama": {
        "base_url": os.getenv("OLLAMA_BASE_URL", "http://localhost:11434/v1"),
        "model":    os.getenv("OLLAMA_MODEL", "gemma4"),
        "api_key":  os.getenv("OLLAMA_API_KEY", "ollama"),  # ignored by Ollama, required by the client
    },
}

def live_llm(mode: str, temperature: float = 0.2):
    """Build a chat client for a local OpenAI-compatible server (LM Studio or Ollama).
    langchain_openai is imported lazily so the offline path needs no extra import."""
    from langchain_openai import ChatOpenAI
    cfg = BACKENDS[mode]
    return ChatOpenAI(
        model=cfg["model"], temperature=temperature,
        timeout=45, max_retries=0,
        base_url=cfg["base_url"], api_key=cfg["api_key"],
    )

def model(scripted: list[str], temperature: float = 0.2):
    """Return a live client when MODE names a live backend, else a deterministic
    model that replays `scripted`. MODE defaults to offline so checks stay reproducible."""
    if MODE == "offline":
        return FakeListChatModel(responses=scripted)
    if MODE in BACKENDS:
        return live_llm(MODE, temperature)
    raise ValueError(f"Unknown MODE {MODE!r}. Use 'offline', 'lmstudio', or 'ollama'.")

def approx_tokens(text: str) -> int:
    """Cheap, model-free token estimate: ~4 chars per token."""
    return 0 if not text else max(1, round(len(text) / 4))

def messages_tokens(msgs: list[BaseMessage]) -> int:
    return sum(approx_tokens(m.content) for m in msgs)

def contains_fuzzy(hay: str, needle: str, thresh: int = 80) -> bool:
    return fuzz.partial_ratio((hay or "").lower(), (needle or "").lower()) >= thresh

def ctx_has(msgs: list[BaseMessage], needle: str) -> bool:
    return needle.lower() in " ".join(m.content for m in msgs).lower()

def timed(fn):
    def wrap(*a, **k):
        t0 = time.perf_counter(); out = fn(*a, **k); return out, time.perf_counter() - t0
    return wrap

print("Model backend ready. MODE =", MODE)

In [ ]:
# A4: soft check harness (counts results, never raises)
_RESULTS = {"pass": 0, "fail": 0}

def check(label: str, cond) -> bool:
    """cond may be a bool or a zero-arg callable. Exceptions count as a failure, not a crash."""
    try:
        ok = cond() if callable(cond) else cond
        ok = True if ok is None else bool(ok)
        note = ""
    except NotImplementedError:
        ok, note = False, "  (not implemented yet)"
    except Exception as e:
        ok, note = False, f"  ({type(e).__name__}: {e})"
    _RESULTS["pass" if ok else "fail"] += 1
    print(f"{'PASS' if ok else 'FAIL'}  {label}{note}")
    return ok

def score():
    p, f = _RESULTS["pass"], _RESULTS["fail"]
    print(f"\nSCORE: {p} passed / {f} failed / {p + f} total")

def load_history(mem):
    """Replay HISTORY into any memory object exposing add_user / add_ai."""
    for t in HISTORY:
        (mem.add_user if t["role"] == "user" else mem.add_ai)(t["text"])

print("check() ready")

---
## Part B: Buffer memory (and a windowed variant)

**Buffer memory** is the baseline: keep every message and re-send the whole list each turn. Perfect recall, but tokens and latency grow without bound. A **window** caps that by keeping only the last *k* exchanges, which trades old-fact recall for a fixed cost.

**TODO 1.** Implement `BufferMemory` and `windowed()` so the checks below pass.

In [ ]:
# B: SOLUTION
from dataclasses import dataclass, field

@dataclass
class BufferMemory:
    """Full-history memory. Stores every turn; context() re-sends all of it."""
    system: str = SYSTEM_PROMPT
    messages: list = field(default_factory=list)

    def add_user(self, text: str) -> None:
        self.messages.append(HumanMessage(content=text))

    def add_ai(self, text: str) -> None:
        self.messages.append(AIMessage(content=text))

    def context(self) -> list[BaseMessage]:
        """System prompt followed by the entire message history, in order."""
        return [SystemMessage(content=self.system)] + self.messages

def windowed(mem: BufferMemory, k_exchanges: int) -> list[BaseMessage]:
    """Keep the system message plus only the last k_exchanges user/assistant pairs."""
    return trim_messages(
        mem.context(),
        token_counter=lambda ms: len(ms),   # count messages, not tokens
        max_tokens=1 + 2 * k_exchanges,     # system + k pairs
        strategy="last",
        include_system=True,
        start_on="human",
    )

In [ ]:
# B: checks
def _b_recall():
    buf = BufferMemory(); load_history(buf)
    ctx = buf.context()
    print(f"buffer context: {len(ctx)} messages, ~{messages_tokens(ctx)} tokens")
    return all(ctx_has(ctx, t) for t in
               EV["prefs_tokens"] + [EV["ticket"], EV["user"], EV["device"]])

def _b_window():
    buf = BufferMemory(); load_history(buf)
    win = windowed(buf, k_exchanges=1)
    print("  window(k=1):", [type(m).__name__ for m in win])
    return (len(win) == 3
            and ctx_has(win, EV["ticket"])          # recent fact kept
            and not ctx_has(win, EV["user"])        # early fact dropped
            and not ctx_has(win, "H.265"))          # early fact dropped

check("B1 buffer keeps every fact (full recall)", _b_recall)
check("B2 window(k=1) keeps recent, drops old", _b_window)

In [ ]:
# B: see it answer the final question (offline by default; set MODE to a live backend for a real model)
try:
    buf = BufferMemory(); load_history(buf)
    prompt_msgs = buf.context() + [HumanMessage(content=FINAL_Q)]
    llm = model(scripted=["You preferred H.265 at 20 Mbps, and your last ticket was A-8821."])
    reply, dt = timed(llm.invoke)(prompt_msgs)
    print(f"[buffer] {reply.content}   ({dt*1000:.0f} ms, ~{messages_tokens(prompt_msgs)} ctx tokens)")
    for tok in EV["expected_answer_contains"]:
        print(f"   recalled {tok!r}: {contains_fuzzy(reply.content, tok)}")
except NotImplementedError:
    print("Implement TODO 1 first, then re-run this cell.")

---
## Part C: Summary memory

Instead of carrying every message, keep the **last few verbatim** and fold everything older into a running **summary** (produced by the model). Context size stays roughly flat as the conversation grows, at the cost of losing verbatim detail.

**TODO 2.** Implement `SummaryMemory`. When more than `keep_recent` messages are held, roll the oldest into the summary.

In [ ]:
# C: SOLUTION
@dataclass
class SummaryMemory:
    """Keep the last `keep_recent` messages verbatim; summarize everything older."""
    llm: object
    system: str = SYSTEM_PROMPT
    keep_recent: int = 2
    summary: str = ""
    recent: list = field(default_factory=list)

    def _summarize(self, older: list[BaseMessage]) -> str:
        transcript = "\n".join(f"{m.type}: {m.content}" for m in older)
        prompt = (f"Prior summary:\n{self.summary or '(none)'}\n\n"
                  f"New lines:\n{transcript}\n\nReturn one concise updated summary.")
        return self.llm.invoke(prompt).content.strip()

    def _add(self, msg: BaseMessage) -> None:
        self.recent.append(msg)
        if len(self.recent) > self.keep_recent:
            older, self.recent = self.recent[:-self.keep_recent], self.recent[-self.keep_recent:]
            self.summary = self._summarize(older)

    def add_user(self, text: str) -> None: self._add(HumanMessage(content=text))
    def add_ai(self, text: str) -> None:   self._add(AIMessage(content=text))

    def context(self) -> list[BaseMessage]:
        head = self.system + (f"\n\nConversation summary so far:\n{self.summary}" if self.summary else "")
        return [SystemMessage(content=head)] + self.recent

In [ ]:
# C: checks
# Scripted summaries stand in for the model offline; the last one carries the ticket.
SUMMARY_SCRIPT = [
    "Jordan uses the video editor on a MacBook Pro.",
    "Jordan hits 4K export failures on app v2.3 / M1 Pro; prefers H.265 at 20 Mbps.",
    "Jordan prefers H.265 at 20 Mbps; last support ticket A-8821.",
]
def _c_compact():
    sm = SummaryMemory(llm=model(scripted=SUMMARY_SCRIPT), keep_recent=2)
    load_history(sm)
    buf = BufferMemory(); load_history(buf)
    print(f"  summary: {len(sm.context())} msgs (~{messages_tokens(sm.context())} tok) "
          f"vs buffer {len(buf.context())} msgs (~{messages_tokens(buf.context())} tok)")
    print(f"  running summary: {sm.summary!r}")
    return len(sm.context()) < len(buf.context())

def _c_retains_ticket():
    sm = SummaryMemory(llm=model(scripted=SUMMARY_SCRIPT), keep_recent=2)
    load_history(sm)
    return EV["ticket"].lower() in sm.summary.lower()

check("C1 summary is more compact than buffer", _c_compact)
check("C2 summary retains the ticket id", _c_retains_ticket)

---
## Part D: Entity memory

Keep a small **side table of facts** ("entities") instead of transcript. On each user turn, ask the model to extract stable facts as JSON, merge them into a dict (latest value wins), and inject the table into the system prompt. Tiny, durable context; great for profile-style recall.

**TODO 3.** Implement `EntityMemory.observe()` and `facts_block()`.

In [ ]:
# D: SOLUTION
@dataclass
class EntityMemory:
    """Extract stable facts per user turn into a dict; inject them into the prompt."""
    llm: object
    system: str = SYSTEM_PROMPT
    store: dict = field(default_factory=dict)
    recent: list = field(default_factory=list)

    @staticmethod
    def _parse(raw: str) -> dict:
        s = raw.strip()
        if s.startswith("```"):
            s = s.split("```")[1]
            if s.startswith("json"):
                s = s[4:]
        try:
            return json.loads(s)
        except Exception:
            return {}

    def observe(self, text: str) -> None:
        self.recent.append(HumanMessage(content=text))
        prompt = ("Extract stable facts about the user/product as flat JSON "
                  "(snake_case keys). If none, return {}.\n\nText: " + text)
        for k, v in self._parse(self.llm.invoke(prompt).content).items():
            if v:
                self.store[k] = v   # latest value wins

    def add_ai(self, text: str) -> None:
        self.recent.append(AIMessage(content=text))

    def facts_block(self) -> str:
        if not self.store:
            return self.system
        lines = "\n".join(f"- {k}: {v}" for k, v in self.store.items())
        return self.system + "\n\nKnown facts about the user:\n" + lines

    def context(self, keep_recent: int = 2) -> list[BaseMessage]:
        # Facts are durable, so we only need to carry the last couple of turns.
        return [SystemMessage(content=self.facts_block())] + self.recent[-keep_recent:]

In [ ]:
# D: checks
# One scripted extraction per user turn (offline). Live mode lets the model do this.
ENTITY_SCRIPT = [
    '{"user":"Jordan","device":"MacBook Pro","product":"video editor"}',
    '{"app_version":"v2.3","chip":"M1 Pro"}',
    '{"export_prefs":"H.265 at 20 Mbps"}',
    '{"ticket":"A-8821"}',
]
def _d_store():
    em = EntityMemory(llm=model(scripted=ENTITY_SCRIPT))
    for t in HISTORY:
        em.observe(t["text"]) if t["role"] == "user" else em.add_ai(t["text"])
    print("  store:", em.store)
    want = {"user": "Jordan", "device": "MacBook Pro",
            "export_prefs": "H.265 at 20 Mbps", "ticket": "A-8821"}
    return all(em.store.get(k) == v for k, v in want.items())

def _d_block():
    em = EntityMemory(llm=model(scripted=ENTITY_SCRIPT))
    for t in HISTORY:
        em.observe(t["text"]) if t["role"] == "user" else em.add_ai(t["text"])
    return EV["ticket"] in em.facts_block()

check("D1 entity store captures user/device/prefs/ticket", _d_store)
check("D2 facts_block injects the ticket", _d_block)

---
## Part E: Privacy, and when *not* to use memory

Memory that stores raw user text also stores raw PII. **Redact before you store**: the order matters exactly like scrub-before-dedup in a data pipeline. And sometimes the right amount of memory is *none*. For a one-shot task you can hand the model an explicit transcript and forbid it from relying on anything else.

**TODO 4.** Implement `redact()` and `answer_without_memory()`.

In [ ]:
# E: SOLUTION
_PII = [
    (re.compile(r"\bA-\d{4,6}\b"), "<TICKET>"),
    (re.compile(r"\b[\w.+-]+@[\w-]+\.[a-z]{2,}\b", re.I), "<EMAIL>"),
    (re.compile(r"\b\d{4}-\d{2}-\d{2}\b"), "<DATE>"),
]
def redact(text: str) -> str:
    """Replace ticket ids, emails, and ISO dates with placeholder tokens."""
    t = text or ""
    for pat, tok in _PII:
        t = pat.sub(tok, t)
    return t

def answer_without_memory(llm, transcript: str, question: str) -> str:
    """Answer using only the supplied transcript; forbid reliance on prior memory."""
    prompt = ("Answer ONLY from the transcript below; do not rely on prior memory.\n\n"
              f"TRANSCRIPT:\n{transcript}\n\nQUESTION: {question}")
    return llm.invoke(prompt).content

In [ ]:
# E: checks
def _e_redact():
    red = redact("My last support ticket was #A-8821 emailed to j@x.com on 2025-03-14")
    print("  redacted:", red)
    return all(x not in red for x in ["A-8821", "j@x.com", "2025-03-14"])

def _e_scrub_before_store():
    # scrub-before-store: redacted buffer can no longer recall the ticket
    buf = BufferMemory()
    for t in HISTORY:
        (buf.add_user if t["role"] == "user" else buf.add_ai)(redact(t["text"]))
    return not ctx_has(buf.context(), EV["ticket"])

def _e_no_memory():
    transcript = "\n".join(f"{t['role']}: {t['text']}" for t in HISTORY)
    llm = model(scripted=["You preferred H.265 at 20 Mbps; last ticket A-8821."])
    ans = answer_without_memory(llm, transcript, FINAL_Q)
    print("  no-memory answer:", ans)
    return all(contains_fuzzy(ans, tok) for tok in EV["expected_answer_contains"])

check("E1 redact removes ticket / email / date", _e_redact)
check("E2 scrub-before-store blocks ticket recall", _e_scrub_before_store)
check("E3 no-memory baseline answers from transcript", _e_no_memory)

---
## Findings: compare the three patterns

The table below is computed live from your implementations. Read it, then write 5 to 8 bullets in the markdown cell that follows: which pattern you would default to for a support assistant, and why.

In [ ]:
# Findings table (computed from your code)
def _profile():
    buf = BufferMemory(); load_history(buf)
    sm = SummaryMemory(llm=model(scripted=SUMMARY_SCRIPT), keep_recent=2); load_history(sm)
    em = EntityMemory(llm=model(scripted=ENTITY_SCRIPT))
    for t in HISTORY:
        em.observe(t["text"]) if t["role"] == "user" else em.add_ai(t["text"])
    rows = [
        ("buffer",  len(buf.context()), messages_tokens(buf.context()), "full verbatim recall; cost grows every turn"),
        ("summary", len(sm.context()),  messages_tokens(sm.context()),  "flat, compact; verbatim detail is lost"),
        ("entity",  len(em.context()),  messages_tokens(em.context()),  "durable facts; recall survives long gaps"),
    ]
    print(f"{'pattern':9s} {'msgs':>5s} {'~tokens':>8s}   note")
    for name, n, tok, note in rows:
        print(f"{name:9s} {n:5d} {tok:8d}   {note}")
try:
    _profile()
except NotImplementedError:
    print("Finish the TODOs above to populate the comparison table.")

### Your findings (edit this cell)

_Replace with 5 to 8 bullets:_

- Buffer vs summary vs entity on **recall**: ...
- Buffer vs summary vs entity on **token cost / latency**: ...
- Where windowing or summarization **loses** a needed fact: ...
- The **privacy** cost of raw storage and how redaction changes recall: ...
- When you would use **no memory** at all: ...
- **Your team's default** for a Cordwell-style support assistant, and why: ...

---
## Stretch goals *(optional)*

Two extensions for fast finishers. Solutions are in the instructor key.

**Stretch A: token-budgeted summary.** Subclass `SummaryMemory` so `recent` is bounded by an approximate **token budget** instead of a message count. Roll the oldest recent message into the summary while `messages_tokens(self.recent)` exceeds the budget.

**Stretch B: entity conflict.** Show that `EntityMemory` resolves a changed fact with **latest-wins**: observe a preference, then observe a corrected preference, and confirm the store holds the newer value.

In [ ]:
# Stretch A: SOLUTION
class TokenBudgetSummaryMemory(SummaryMemory):
    def __init__(self, llm, budget_tokens: int = 40, **kw):
        super().__init__(llm=llm, **kw)
        self.budget = budget_tokens
    def _add(self, msg: BaseMessage) -> None:
        self.recent.append(msg)
        while messages_tokens(self.recent) > self.budget and len(self.recent) > 1:
            self.summary = self._summarize([self.recent.pop(0)])

def _sa():
    tb = TokenBudgetSummaryMemory(llm=model(scripted=SUMMARY_SCRIPT * 2), budget_tokens=40)
    load_history(tb)
    print(f"  recent ~{messages_tokens(tb.recent)} tokens (budget 40); summary set: {bool(tb.summary)}")
    return messages_tokens(tb.recent) <= 40 and bool(tb.summary)
check("Stretch A budget holds recent within token budget", _sa)

In [ ]:
# Stretch B: SOLUTION (entity latest-wins)
def _sb():
    em = EntityMemory(llm=model(scripted=[
        '{"export_prefs":"H.265 at 20 Mbps"}',
        '{"export_prefs":"H.264 at 12 Mbps"}',
    ]))
    em.observe("I prefer H.265 exports with 20 Mbps.")
    first = em.store["export_prefs"]
    em.observe("Actually, switch me to H.264 at 12 Mbps.")
    print(f"  before: {first!r}  ->  after: {em.store['export_prefs']!r}")
    return first == "H.265 at 20 Mbps" and em.store["export_prefs"] == "H.264 at 12 Mbps"
check("Stretch B entity store is latest-wins", _sb)

---
## Acceptance

Run the cell below. For a complete lab it should report **11 passed / 0 failed** (9 core checks plus 2 stretch). If you skipped the stretch goals, expect **9 passed**.

In [ ]:
score()